### Feature Engineering

This notebook takes retail_clean.csv and produces new analytical tables and calculcated fields to support calculating the identified KPIs and Metrics

### Import libraries and create path to dataset

In [10]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

### Rolling 12-Month YoY Growth

This table calculates each category's YoY growth as percentage, showing each category's performance over the past 12 months.

Viewing growth over a 12 month period minimizes the impact of seasonality, as most categories show a distinct increase in turnover towards December.

In [13]:
# Read clean retail CSV into dataframe
df_all = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv', parse_dates=['date'])
df_all = df_all.sort_values(['category', 'date'])

# Save Total (Industry) into separate dataframe
df_total = df_all[df_all['category'] == 'Total (Industry)'].copy()

# Remove Total (Industry) from working dataframe
df = df_all[df_all['category'] != 'Total (Industry)'].copy()

# Sum the last 12 months of turnover, grouped by category
# min_periods=12 means don't calculate until we have a full 12 months of data
df['r12m_turnover'] = (
    df.groupby('category')['turnover_m']
    .transform(lambda x: x.rolling(12, min_periods=12).sum())
)

# Calculate the prior 12 months of retail turnover by shifting 12 rows down
df['r12m_prior'] = (
    df.groupby('category')['r12m_turnover']
    .shift(12)
)

# YoY growth = (this year - last year) / last year × 100
df['yoy_growth_pct'] = (
    (df['r12m_turnover'] - df['r12m_prior']) / df['r12m_prior'] * 100
    ).round(1)

df.to_csv(DATA_PROCESSED / 'retail_yoy_growth.csv', index=False)
print('Saved: retail_yoy_growth.csv')

Saved: retail_yoy_growth.csv


### Retail Benchmark

This table calculates the total industry rolling YoY growth, serving as a benchmark every category can be measured against. 

In [ ]:
# Read and sort data from df containing Total (Industry) only
df_total = df_total.sort_values('date')

# Calculate rolling 12 month turnover
df_total['r12m_turnover'] = df_total['turnover_m'].rolling(12, min_periods=12).sum()

# Calculate prior 12 month rolling turnover
df_total['r12m_prior'] = df_total['r12m_turnover'].shift(12)

# Calculate YoY growth as percent
df_total['yoy_growth_pct'] = ( (df_total['r12m_turnover'] - df_total['r12m_prior']) / df_total['r12m_prior'] * 100
    ).round(1)

# Save relevant columns to retail_benchmark df
retail_benchmark = df_total[['date', 'turnover_m', 'r12m_turnover', 'yoy_growth_pct']].copy()

# Save data to CSV
retail_benchmark.to_csv(DATA_PROCESSED / 'retail_benchmark.csv', index=False)
print('Saved: retail_benchmark.csv')

Saved: retail_benchmark.csv
           date  turnover_m  r12m_turnover  yoy_growth_pct
3114 1982-04-01      3396.4            NaN             NaN
3115 1982-05-01      3497.9            NaN             NaN
3116 1982-06-01      3357.8            NaN             NaN
3117 1982-07-01      3486.8            NaN             NaN
3118 1982-08-01      3355.9            NaN             NaN
3119 1982-09-01      3454.3            NaN             NaN
3120 1982-10-01      3551.5            NaN             NaN
3121 1982-11-01      3830.5            NaN             NaN
3122 1982-12-01      5179.7            NaN             NaN
3123 1983-01-01      3384.5            NaN             NaN
3124 1983-02-01      3369.8            NaN             NaN
3125 1983-03-01      3805.3        43670.4             NaN
3126 1983-04-01      3665.1        43939.1             NaN
3127 1983-05-01      3760.0        44201.2             NaN
3128 1983-06-01      3630.8        44474.2             NaN
3129 1983-07-01      3686.5 

### Retail Category Ranking

This table ranks each category's YoY growth in comparison the Total (Industry) benchmark, giving insight into which categories are performing above or below the overall industry.

In [17]:
# Pull the latest benchmark data from the Total (Industry)
benchmark = retail_benchmark[ retail_benchmark['date'] == retail_benchmark['date'].max()]['yoy_growth_pct'].values[0]

# Get the latest rolling YoY growth per category from retail_yoy_growth
latest = df[df['date'] == df['date'].max()].copy()
latest['benchmark_growth_pct'] = benchmark

# gap = ppt above or below benchmark — negative means underperforming
latest['gap_vs_national_ppt'] = (latest['yoy_growth_pct'] - benchmark).round(1)
latest = latest.sort_values('yoy_growth_pct') # worst performers at top

# Save relevant columns to retail_category_ranking df
retail_category_ranking = latest[['category', 'yoy_growth_pct', 'benchmark_growth_pct', 'gap_vs_national_ppt']].copy()

# Save data to CSV
retail_category_ranking.to_csv(DATA_PROCESSED / 'retail_category_ranking.csv', index=False)
print('Saved: retail_category_ranking.csv')

Saved: retail_category_ranking.csv
